# Titanic: Data Collection, Cleaning & Exploratory Data Analysis

**Goal:** turn a raw, messy dataset into clean data and find patterns in who survived the Titanic disaster.

**Dataset:** Titanic passenger data (891 passengers), the training file from Kaggle's *Titanic: Machine Learning from Disaster* competition. A public CSV copy is used for loading.

| Column | Meaning |
|---|---|
| PassengerId | Row identifier |
| Survived | 0 = died, 1 = survived |
| Pclass | Ticket class (1 = upper, 2 = middle, 3 = lower) |
| Name | Passenger name (includes title such as Mr, Mrs) |
| Sex | male / female |
| Age | Age in years |
| SibSp | Number of siblings or spouses aboard |
| Parch | Number of parents or children aboard |
| Ticket | Ticket number |
| Fare | Ticket price |
| Cabin | Cabin number |
| Embarked | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Works both inside the repo (notebooks/ folder) and in Google Colab
ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
IMG_DIR = f"{ROOT}/images"
DATA_DIR = f"{ROOT}/data"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

def save_fig(name):
    """Save the current figure to images/ and show it."""
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/{name}.png", dpi=150)
    plt.show()

## 2. Data collection (loading)

In [ ]:
RAW_PATH = f"{DATA_DIR}/titanic.csv"
URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if os.path.exists(RAW_PATH):
    raw = pd.read_csv(RAW_PATH)          # use the local copy if it exists
else:
    raw = pd.read_csv(URL)               # otherwise download it
    raw.to_csv(RAW_PATH, index=False)    # and keep a raw copy in data/

print("Rows, columns:", raw.shape)
raw.head()

## 3. Initial inspection

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all").T

## 4. Data quality checks
Before cleaning, look for missing values, duplicates and inconsistent or impossible values.

In [ ]:
missing = raw.isna().sum().to_frame("missing")
missing["percent"] = (missing["missing"] / len(raw) * 100).round(1)
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)
missing

In [ ]:
plt.figure(figsize=(6, 3.5))
sns.barplot(x=missing.index, y=missing["percent"], color="steelblue")
plt.ylabel("% missing")
plt.xlabel("Column")
plt.title("Missing values by column")
save_fig("missing_values")

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
print("Duplicate PassengerId:", raw["PassengerId"].duplicated().sum())
print("Age <= 0 or > 100:", ((raw["Age"] <= 0) | (raw["Age"] > 100)).sum())
print("Fare == 0:", (raw["Fare"] == 0).sum())
print("Negative fares:", (raw["Fare"] < 0).sum())
print()
for col in ["Sex", "Embarked", "Pclass", "Survived"]:
    print(raw[col].value_counts(dropna=False).to_string(), "\n")

## 5. Data cleaning

In [ ]:
df = raw.copy()          # never modify the raw data
cleaning_log = []        # record every step

# 1. Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
cleaning_log.append(("Removed duplicate rows", before - len(df)))

# 2. Standardise text values (strip spaces, consistent case)
df["Sex"] = df["Sex"].str.strip().str.lower()
df["Embarked"] = df["Embarked"].str.strip().str.upper()

# 3. Embarked: fill the few missing values with the most common port
n = df["Embarked"].isna().sum()
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
cleaning_log.append(("Filled missing Embarked with mode", n))

# 4. Age: fill missing values with the median age of the passenger's class and sex
n = df["Age"].isna().sum()
df["Age"] = df["Age"].fillna(df.groupby(["Pclass", "Sex"])["Age"].transform("median"))
df["Age"] = df["Age"].fillna(df["Age"].median())   # safety net for any empty group
cleaning_log.append(("Filled missing Age with median by Pclass and Sex", n))

# 5. Fare: fill any missing value with the median fare of the class
n = df["Fare"].isna().sum()
df["Fare"] = df["Fare"].fillna(df.groupby("Pclass")["Fare"].transform("median"))
cleaning_log.append(("Filled missing Fare with median by Pclass", n))

# 6. Cabin: too many missing values (about 77%) to fill, so keep only whether a cabin is known
df["HasCabin"] = df["Cabin"].notna().astype(int)
df = df.drop(columns="Cabin")
cleaning_log.append(("Replaced Cabin with HasCabin flag and dropped Cabin", None))

# 7. Feature engineering from Name, SibSp and Parch
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["Title"] = df["Title"].where(df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare").fillna("Rare")
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 35, 60, 120],
                        labels=["Child", "Teen", "Young adult", "Adult", "Senior"])
df["Survival"] = df["Survived"].map({0: "Died", 1: "Survived"})
cleaning_log.append(("Created Title, FamilySize, IsAlone, AgeGroup, Survival", None))

# 8. Drop columns that are identifiers with little analytical value
df = df.drop(columns=["Name", "Ticket"])
cleaning_log.append(("Dropped Name and Ticket", None))

# 9. Set suitable data types
for col in ["Sex", "Embarked", "Title"]:
    df[col] = df[col].astype("category")

pd.DataFrame(cleaning_log, columns=["Step", "Rows affected"])

In [ ]:
# Fare outliers: flagged, not removed (expensive tickets are real, not errors)
q1, q3 = df["Fare"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"Fares above {upper:.1f} (IQR rule): {(df['Fare'] > upper).sum()} passengers (kept)")

# Final validation
print("Missing values left:", int(df.isna().sum().sum()))
print("Shape after cleaning:", df.shape)
df.dtypes

In [ ]:
df.to_csv(f"{DATA_DIR}/titanic_cleaned.csv", index=False)
df.head()

## 6. Exploratory data analysis

### 6.1 Overall survival

In [ ]:
rate = df["Survived"].mean()
print(f"Overall survival rate: {rate:.1%} ({df['Survived'].sum()} of {len(df)} passengers)")

plt.figure(figsize=(4.5, 3.5))
sns.countplot(x="Survival", data=df, order=["Died", "Survived"], color="steelblue")
plt.title("Passengers who died vs survived")
save_fig("survival_counts")

### 6.2 Distributions of single variables

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 8))
sns.histplot(df["Age"], bins=30, ax=ax[0, 0], color="steelblue")
ax[0, 0].set_title("Age distribution")
sns.histplot(df["Fare"], bins=40, ax=ax[0, 1], color="steelblue")
ax[0, 1].set_title("Fare distribution (right-skewed)")
sns.countplot(x="Pclass", data=df, ax=ax[1, 0], color="steelblue")
ax[1, 0].set_title("Passengers by class")
sns.countplot(x="Sex", data=df, ax=ax[1, 1], color="steelblue")
ax[1, 1].set_title("Passengers by sex")
save_fig("distributions")

### 6.3 Survival rate by group

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, col in zip(ax.flat, ["Sex", "Pclass", "Embarked", "AgeGroup", "FamilySize", "HasCabin"]):
    sns.barplot(x=col, y="Survived", data=df, errorbar=None, ax=a, color="steelblue")
    a.set_ylim(0, 1)
    a.set_ylabel("Survival rate")
    a.set_title(f"Survival rate by {col}")
save_fig("survival_by_group")

In [ ]:
pivot = df.pivot_table(values="Survived", index="Sex", columns="Pclass", observed=True)
plt.figure(figsize=(5.5, 3.5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Survival rate by sex and class")
save_fig("survival_sex_class")
pivot.round(3)

### 6.4 Age, fare and class

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.kdeplot(data=df, x="Age", hue="Survival", common_norm=False, fill=True, ax=ax[0])
ax[0].set_title("Age distribution: died vs survived")
sns.boxplot(x="Pclass", y="Fare", data=df, ax=ax[1], color="lightsteelblue")
ax[1].set_yscale("log")
ax[1].set_title("Fare by class (log scale)")
save_fig("age_fare_class")

### 6.5 Correlations

In [ ]:
num_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "HasCabin"]
plt.figure(figsize=(7, 5.5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
save_fig("correlation_heatmap")

In [ ]:
# Exact numbers behind the observations below
print(df.groupby("Sex", observed=True)["Survived"].mean().round(3), "\n")
print(df.groupby("Pclass")["Survived"].mean().round(3), "\n")
print(df.groupby("Embarked", observed=True)["Survived"].mean().round(3), "\n")
print(df.groupby("AgeGroup", observed=True)["Survived"].mean().round(3), "\n")
print(df.groupby("FamilySize")["Survived"].agg(["mean", "count"]).round(3), "\n")
print(df.groupby("HasCabin")["Survived"].mean().round(3))

## 7. Key observations

*(Check each point against the outputs above after running the notebook.)*

1. **Overall:** only about 38% of passengers survived.
2. **Sex:** the strongest pattern. Women survived at roughly 74%, men at roughly 19%.
3. **Class:** survival falls with ticket class: roughly 63% (1st), 47% (2nd), 24% (3rd). Higher classes also paid much higher fares.
4. **Sex and class together:** women in 1st and 2nd class almost all survived, while men in 3rd class had the lowest survival rate.
5. **Age:** children had better survival chances than adults, and the oldest passengers did worst.
6. **Family size:** passengers travelling alone or in very large families survived less often than those in small families of 2 to 4.
7. **Embarkation:** passengers from Cherbourg survived more often than those from Southampton, partly because more of them were 1st class.
8. **Cabin:** passengers with a recorded cabin survived much more often than those without, but this mostly reflects higher class.
9. **Data quality:** Age was missing for about 20% of passengers and Cabin for about 77%, so results involving these columns are less certain. Fares are strongly right-skewed.
10. **Correlations:** Survived correlates most with Pclass and Fare (and with sex, which is not in the numeric matrix).